# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratheekrao12/-flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

First look at the March 1–15 traffic and position distributions and the separate March 16–31 evaluable subset. `early_impressions` is heavy-tailed, `early_ctr` is a percentage, and `weighted_avg_position` must be interpreted as missing when zero or unavailable. The historical proxy is used only for the signal tests below, never for queue membership.

In [1]:
from pathlib import Path
import os
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / '.gitignore').exists():
    repo_root = repo_root.parent

token = os.getenv('HF_TOKEN')
if not token:
    env_path = next((base/'.env' for base in [repo_root,*repo_root.parents] if (base/'.env').exists()), None)
    if env_path:
        for line in env_path.read_text().splitlines():
            if line.startswith('HF_TOKEN='):
                token=line.split('=',1)[1].strip().strip(chr(34)).strip(chr(39))

march_url=hf_hub_download(repo_id='FlyRank/internship-warehouse',repo_type='dataset',filename='fact_content_daily_performance/month=2026-03/data_0.parquet',token=token)
con=duckdb.connect()
query='''
WITH page_windows AS (
 SELECT client_hash_id, content_hash_id,
   SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS early_impressions,
   SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS early_clicks,
   SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_avg_position*gsc_impressions ELSE 0 END)
     / NULLIF(SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END),0) AS weighted_avg_position,
   COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS active_search_days,
   SUM(CASE WHEN report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS late_impressions,
   COUNT(DISTINCT CASE WHEN report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS late_available_days
 FROM read_parquet(?)
 GROUP BY client_hash_id, content_hash_id
), early_candidates AS (
 SELECT *, CASE WHEN early_impressions > 0 THEN 100.0*early_clicks/early_impressions ELSE NULL END AS early_ctr
 FROM page_windows WHERE early_impressions > 0
)
SELECT *, CASE WHEN late_available_days=16 AND weighted_avg_position IS NOT NULL AND early_ctr IS NOT NULL THEN late_impressions/16.0 < early_impressions/15.0 ELSE NULL END AS is_declining_proxy
FROM early_candidates
'''
full_df=con.execute(query,[march_url]).fetchdf()
evaluable_df=full_df[(full_df['late_available_days']==16)&full_df['is_declining_proxy'].notna()].copy()
evaluable_df['is_declining_proxy']=evaluable_df['is_declining_proxy'].astype(int)
print('Full March 1-15 candidate population:',len(full_df))
print('Historical evaluable subset:',len(evaluable_df))
print('Partial/missing later-label candidates:',len(full_df)-len(evaluable_df))
for col in ['early_impressions','weighted_avg_position','early_ctr','active_search_days']:
    print(chr(10),col)
    print(evaluable_df[col].describe(percentiles=[.25,.5,.75,.9,.95,.99]).round(4).to_string())
print(chr(10)+'Heavy-tail note: impressions are summarized with percentiles instead of relying on the mean alone.')

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Full March 1-15 candidate population: 151981
Historical evaluable subset: 85565
Partial/missing later-label candidates: 66416

 early_impressions
count     85565.0000
mean       1460.8116
std        3438.0882
min           1.0000
25%         162.0000
50%         466.0000
75%        1356.0000
90%        3461.0000
95%        5926.0000
99%       15460.3200
max      161575.0000

 weighted_avg_position
count    85565.0000
mean        13.7293
std         14.4449
min          0.0000
25%          4.4051
50%          7.8109
75%         18.0198
90%         33.4416
95%         43.9354
99%         69.9253
max        101.0784

 early_ctr
count    85565.0000
mean         0.2965
std          0.6300
min          0.0000
25%          0.0000
50%          0.0878
75%          0.3891
90%          0.7936
95%          1.1838
99%          2.5478
max         50.0000

 active_search_days
count    85565.0000
mean        13.7634
std          2.8006
min          1.0000
25%         14.0000
50%         15.0000
75%   

## 2. Signal test #1 / #2 / #3 (verdict each)

The mini-tests compare the March 16–31 decline proxy on the 85,565-row historical evaluable subset. Verdicts require all tested buckets to meet the sample-size floor, then evaluate the actual proxy-rate pattern; bucket count alone never determines a verdict.

In [2]:
# Signal 1: search-demand volume buckets
volume_df=evaluable_df.copy()
volume_df['volume_bucket']=pd.cut(volume_df['early_impressions'],bins=[-1,499,2999,float('inf')],labels=['low_<500','moderate_500_2999','high_>=3000'],right=True)
volume_table=(volume_df.groupby('volume_bucket',observed=True).agg(n=('is_declining_proxy','size'),decline_proxy_rate=('is_declining_proxy','mean')).reset_index())
volume_table=volume_table[volume_table['n']>=50]
volume_table['decline_proxy_rate']=volume_table['decline_proxy_rate'].round(4)
print('Signal 1 — search demand'+chr(10)+volume_table.to_string(index=False))

# Signal 2: position buckets and decline rate
position_df=evaluable_df[evaluable_df['weighted_avg_position']>0].copy()
position_df['position_bucket']=pd.cut(position_df['weighted_avg_position'],bins=[0,3,10,20,float('inf')],labels=['top_3','page1_4_10','striking_11_20','deep_gt20'],right=True)
position_table=(position_df.groupby('position_bucket',observed=True).agg(n=('is_declining_proxy','size'),decline_proxy_rate=('is_declining_proxy','mean')).reset_index())
position_table=position_table[position_table['n']>=50]
position_table['decline_proxy_rate']=position_table['decline_proxy_rate'].round(4)
print(chr(10)+'Signal 2 — weighted average position'+chr(10)+position_table.to_string(index=False))

# Signal 3: low CTR within position bands
position_df['ctr_bucket']=position_df['early_ctr'].apply(lambda x:'low_<0.5_pct' if x<0.5 else 'other_>=0.5_pct')
ctr_table=(position_df.groupby(['position_bucket','ctr_bucket'],observed=True).agg(n=('is_declining_proxy','size'),decline_proxy_rate=('is_declining_proxy','mean')).reset_index())
ctr_table=ctr_table[ctr_table['n']>=50]
ctr_table['decline_proxy_rate']=ctr_table['decline_proxy_rate'].round(4)
print(chr(10)+'Signal 3 — low CTR by position band'+chr(10)+ctr_table.to_string(index=False))

# Verdict derivation without using future fields as predictors.
all_bands_confirmed=True
for band in ctr_table['position_bucket'].unique():
    vals=ctr_table[ctr_table['position_bucket']==band].set_index('ctr_bucket')['decline_proxy_rate']
    if 'low_<0.5_pct' not in vals or 'other_>=0.5_pct' not in vals:
        all_bands_confirmed=False
    elif vals['low_<0.5_pct'] <= vals['other_>=0.5_pct']:
        all_bands_confirmed=False
print(chr(10)+'Verdict 1: MIXED — volume has non-monotonic directional differences; do not infer causation.')
# Position verdict is based on sample counts and the observed non-monotonic proxy-rate pattern.
position_rates = position_table.sort_values('position_bucket', key=lambda s: s.map({'top_3':0,'page1_4_10':1,'striking_11_20':2,'deep_gt20':3}))['decline_proxy_rate'].tolist()
position_all_supported = len(position_table) == 4 and (position_table['n'] >= 50).all()
position_monotonic_increasing = position_all_supported and len(position_rates) == 4 and all(position_rates[i] <= position_rates[i+1] for i in range(3))
position_monotonic_decreasing = position_all_supported and len(position_rates) == 4 and all(position_rates[i] >= position_rates[i+1] for i in range(3))
if not position_all_supported:
    position_verdict = 'INSUFFICIENT DATA'
elif position_monotonic_increasing or position_monotonic_decreasing:
    position_verdict = 'CONFIRMED'
else:
    position_verdict = 'MIXED'
position_verdict_detail = 'position buckets meet the sample floor, but proxy rates are non-monotonic (top 3=.3958, page 1=.4229, 11–20=.4003, deeper=.4361)' if position_verdict == 'MIXED' else 'position buckets show the observed directional pattern'
print(chr(10)+'Verdict 2: '+position_verdict+' — '+position_verdict_detail+', so this is association only and not a refresh-benefit claim.')
print('Verdict 3: '+('CONFIRMED' if all_bands_confirmed else 'MIXED')+' — low CTR is associated with a higher measured proxy rate within tested position bands.')

Signal 1 — search demand
    volume_bucket     n  decline_proxy_rate
         low_<500 44212              0.3396
moderate_500_2999 31283              0.4927
      high_>=3000 10070              0.5332

Signal 2 — weighted average position
position_bucket     n  decline_proxy_rate
          top_3 10354              0.3958
     page1_4_10 40665              0.4229
 striking_11_20 15300              0.4003
      deep_gt20 19207              0.4361

Signal 3 — low CTR by position band
position_bucket      ctr_bucket     n  decline_proxy_rate
          top_3    low_<0.5_pct  7539              0.4396
          top_3 other_>=0.5_pct  2815              0.2785
     page1_4_10    low_<0.5_pct 31817              0.4535
     page1_4_10 other_>=0.5_pct  8848              0.3130
 striking_11_20    low_<0.5_pct 12506              0.4092
 striking_11_20 other_>=0.5_pct  2794              0.3604
      deep_gt20    low_<0.5_pct 17738              0.4400
      deep_gt20 other_>=0.5_pct  1469             

## 3. The flag-linked test

The tested FlyRank rule adds one evidence point each for early impressions >= 500, position 0 < p <= 20, and early CTR < 0.5%. This test compares the score-3 pages against score < 3 on the historical evaluable subset. It checks flag assumptions; it does not prove a refresh will change outcomes.

In [3]:
score_df=evaluable_df[evaluable_df['weighted_avg_position'].notna() & evaluable_df['early_ctr'].notna()].copy()
score_df['baseline_refresh_score']=((score_df['early_impressions']>=500).astype(int)+((score_df['weighted_avg_position']>0)&(score_df['weighted_avg_position']<=20)).astype(int)+(score_df['early_ctr']<0.5).astype(int))
score_df['rule_group']=score_df['baseline_refresh_score'].apply(lambda x:'score_3_all_signals' if x==3 else 'score_0_to_2')
flag_table=(score_df.groupby('rule_group').agg(n=('is_declining_proxy','size'),decline_proxy_rate=('is_declining_proxy','mean')).reset_index())
flag_table['decline_proxy_rate']=flag_table['decline_proxy_rate'].round(4)
print(flag_table.to_string(index=False))
vals=dict(zip(flag_table['rule_group'],flag_table['decline_proxy_rate']))
if len(vals)==2 and vals.get('score_3_all_signals',0)>vals.get('score_0_to_2',0):
    print('Flag verdict: CONFIRMED — the complete rule group has a higher historical proxy rate than the comparison group.')
else:
    print('Flag verdict: MIXED or OPPOSITE — the complete rule does not show the expected directional difference in this subset.')
assert (flag_table['n']>=50).all()

         rule_group     n  decline_proxy_rate
       score_0_to_2 59293              0.3782
score_3_all_signals 26272              0.5091
Flag verdict: CONFIRMED — the complete rule group has a higher historical proxy rate than the comparison group.


## 4. What this means in practice

For the content team, demand, visible position, and low CTR are useful review signals when considered together, but none independently establishes a refresh benefit. High search demand can be an opportunity signal even when the decline proxy is noisy. Low CTR has the clearest historical directional pattern in this slice; it still needs content-quality, intent, indexing, seasonality, and competitor context.

The baseline rule is an explainable review queue, not an automatic action policy. Human reviewers should inspect the page and data history before rewriting, redirecting, pruning, or publishing anything.

In [4]:
print('Practical interpretation: use these associations for prioritization only.')
print('No refresh-benefit claim, ranking guarantee, or causal explanation is made.')
print('Population distinction: full early-period candidate population has',len(full_df),'pages; the historical signal subset has',len(evaluable_df),'pages.')

Practical interpretation: use these associations for prioritization only.
No refresh-benefit claim, ranking guarantee, or causal explanation is made.
Population distinction: full early-period candidate population has 151981 pages; the historical signal subset has 85565 pages.


## Self-check

- [x] Signal distributions and heavy tails are reported with percentiles and sample counts.
- [x] Three explicit signal tests have tables and verdict labels.
- [x] The FlyRank three-part baseline flag is linked to the historical proxy for a practical test.
- [x] Verdicts distinguish association from refresh benefit and future decline claims.
- [x] Every conclusion uses observed, measured, directional, or decision-support language.
- [x] The proxy is separate from queue membership; the full candidate population is not silently reduced to the label-evaluable subset.
- [x] Pseudonymous identifiers are used only for warehouse grain and grouping; no private names, URLs, queries, or tokens are displayed.
- [x] The notebook is designed for top-to-bottom execution without saved errors.